# 06 - Synthetic 36-month history (standalone, for the Long-term Trends dashboard)
Writes two SYNTHETIC tables to `classic_stable_l0m1dn_catalog.support_command_center` (not produced by the streaming pipeline):
* `gold_hist_kpi_monthly` - Nov 2023 to Oct 2026, month x region: subscribers, streams, tickets, CSAT, SLA, first response, repeat contact rate, incidents, signal-to-declare minutes, customer-detected minutes, payment success, revenue at risk.
* `gold_hist_churn_monthly` - month x region x paid tier: active subscribers, cancellations, churn rate, cancellations by reason, MRR lost.

Story: a synthetic "Command Center launched" milestone in Jan 2025, after which repeat contacts, signal-to-declare time and churn improve gradually. Oct 2026 is aligned with the live 15-day data (CSAT ~4.0, SLA ~80%, repeat contacts ~8%).

In [0]:
import numpy as np, pandas as pd
from pyspark.sql import functions as F
FQ = "classic_stable_l0m1dn_catalog.support_command_center"
rng = np.random.default_rng(36)
reg = spark.table(f"{FQ}.regions").toPandas().groupby("region_group").subscribers.sum()
months = pd.date_range("2023-11-01", "2026-10-01", freq="MS"); LAUNCH = pd.Timestamp("2025-01-01")
rows, crow = [], []
TIER_SHARE = {"Fan": 0.50, "Mega Fan": 0.40, "Ultimate Fan": 0.10}; PRICE = {"Fan": 7.99, "Mega Fan": 11.99, "Ultimate Fan": 15.99}
TIER_CHURN = {"Fan": 1.15, "Mega Fan": 0.9, "Ultimate Fan": 0.7}
for m in months:
    ago = (months[-1].year - m.year) * 12 + (months[-1].month - m.month)
    post = m >= LAUNCH; ramp = np.clip(((m - LAUNCH).days / 365) if post else 0, 0, 1.6) / 1.6   # 0 before launch -> 1 by mid-2026
    season = 1.08 if m.month in (1, 4, 7, 10) else (1.0 if m.month not in (6, 12) else 0.97)    # anime season launches
    n_inc = rng.poisson(5.5)
    for rg, cur in reg.items():
        subs = cur * (1.22 ** (-ago / 12)) * rng.lognormal(0, 0.01)
        streams = subs * 0.075 * season * rng.lognormal(0, 0.02)
        tickets = subs / 1000 * 15.2 * season * (1.12 - 0.12 * ramp) * rng.lognormal(0, 0.03)
        inc = int(rng.binomial(n_inc, 0.3))
        csat = 3.74 + 0.30 * ramp - 0.03 * inc + rng.normal(0, 0.03)
        sla = 0.715 + 0.09 * ramp - 0.006 * inc + rng.normal(0, 0.008)
        frt = 43 - 12 * ramp + rng.normal(0, 1.5)
        rep = 0.142 - 0.059 * ramp + 0.003 * inc + rng.normal(0, 0.004)
        s2d = 88 - 50 * ramp + rng.normal(0, 6) if inc else None
        cdm = inc * (95 - 55 * ramp) * rng.lognormal(0, 0.25)
        pay = 0.976 + 0.004 * ramp - 0.0015 * inc + rng.normal(0, 0.0008)
        rar = subs * 0.0021 * (1 - pay) / 0.024 * (1 + 0.4 * inc) * rng.lognormal(0, 0.2)
        rows.append(dict(month=m.date(), region_group=rg, subscribers=int(subs), peak_concurrent_streams=int(streams), tickets=int(tickets),
                         contacts_per_1k_subs=round(tickets / subs * 1000, 2), avg_csat=round(csat, 3), sla_met_rate=round(sla, 4), avg_first_response_min=round(frt, 1),
                         repeat_contact_rate=round(rep, 4), incidents=inc, avg_signal_to_declare_min=None if s2d is None else round(s2d, 1),
                         customer_detected_minutes=int(cdm), payment_success_rate=round(pay, 4), revenue_at_risk_usd=round(rar, 0), command_center_live=bool(post), is_synthetic=True))
        paid = subs * 0.70
        base = 0.0425 - 0.0055 * ramp + 0.0012 * inc + (0.002 if m.month in (6, 12) else 0)
        for t, sh in TIER_SHARE.items():
            act = int(paid * sh); rate = max(0.01, base * TIER_CHURN[t] + rng.normal(0, 0.0012)); canc = int(act * rate)
            tech = 0.17 - 0.07 * ramp + 0.01 * inc; pay_r = 0.15 - 0.04 * ramp
            mix = np.array([0.30, 0.24, pay_r, tech]); mix = np.append(mix, max(0.05, 1 - mix.sum())); mix = mix / mix.sum()
            parts = (mix * canc).round().astype(int)
            crow.append(dict(month=m.date(), region_group=rg, tier=t, active_subscribers=act, cancellations=canc, churn_rate=round(canc / act, 5),
                             cancel_price=int(parts[0]), cancel_content=int(parts[1]), cancel_payment_failure=int(parts[2]), cancel_playback_tech=int(parts[3]), cancel_other=int(parts[4]),
                             mrr_lost_usd=round(canc * PRICE[t], 2), command_center_live=bool(post), is_synthetic=True))
spark.createDataFrame(pd.DataFrame(rows)).withColumn("month", F.to_date("month")).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.gold_hist_kpi_monthly")
spark.createDataFrame(pd.DataFrame(crow)).withColumn("month", F.to_date("month")).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.gold_hist_churn_monthly")
spark.sql(f"COMMENT ON TABLE {FQ}.gold_hist_kpi_monthly IS 'SYNTHETIC demo history: 36 months x region of support and platform KPIs. Command Center launch milestone Jan 2025 (synthetic). Standalone, not produced by the pipeline.'")
spark.sql(f"COMMENT ON TABLE {FQ}.gold_hist_churn_monthly IS 'SYNTHETIC demo history: 36 months x region x paid tier of active subscribers, cancellations, churn rate, cancellation reasons and MRR lost. Standalone, not produced by the pipeline.'")
print(spark.table(f"{FQ}.gold_hist_kpi_monthly").count(), spark.table(f"{FQ}.gold_hist_churn_monthly").count())